# Lekcja 3.2 — eksperymenty: przycinanie alfa-beta

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie i zwraca jego wynik, liczbę odwiedzonych pozycji i czas w sekundach, tak jak w lekcji 3.1. Funkcja `ordered(cells)` tworzy kółko i krzyżyk, w którym `game.actions` podaje wolne pola w kolejności z listy `cells`. Reguły gry się nie zmieniają, zmienia się tylko kolejność, w jakiej przeszukiwanie ogląda ruchy. Minimax i alfa-beta z nierównościami ostrymi pochodzą z wersji wzorcowej kursu.

In [ ]:
import random
import time
from statistics import mean

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from introai.games import TIC_TAC_TOE, KInARow, reference
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'
CELLS = [(row, col) for row in range(3) for col in range(3)]


def run(search, game, state, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


class Ordered(KInARow):
    """Tic-tac-toe whose moves come in a fixed order of the cells."""

    def __init__(self, cells):
        super().__init__(3, 3, 3)
        self.rank = {cell: i for i, cell in enumerate(cells)}

    def actions(self, state):
        return sorted(super().actions(state), key=self.rank.__getitem__)


def ordered(cells):
    return Ordered(cells)


def strict_alpha_beta(game, state):
    """The reference alpha-beta that cuts off only at v > beta and v < alpha."""
    return reference.alpha_beta_search(game, state, strict=True)


def empty_cells(state):
    return sum(row.count(".") for row in state)


def random_position(empty, seed):
    """A position with `empty` empty cells after random moves, or None if the game ended."""
    rng = random.Random(seed)
    state = TIC_TAC_TOE.initial
    while empty_cells(state) > empty and not TIC_TAC_TOE.is_terminal(state):
        state = TIC_TAC_TOE.result(state, rng.choice(TIC_TAC_TOE.actions(state)))
    return None if TIC_TAC_TOE.is_terminal(state) else state

## Eksperyment 1: ile pozycji z pustej planszy

Cztery przeszukiwania szukają pierwszego ruchu na pustej planszy: minimax, Twój alfa-beta, Twój alfa-beta na planszy, na której najpierw ogląda środek, potem rogi, a na końcu boki, i wzorcowy alfa-beta z nierównościami ostrymi.

**Twoje przewidywanie.** Ile razy mniej pozycji niż minimax odwiedzi alfa-beta? Która kolejność pól będzie lepsza? Ile kosztują nierówności ostre?

*Moje przewidywanie:* …

In [ ]:
centre_first = ordered([(1, 1), (0, 0), (0, 2), (2, 0), (2, 2), (0, 1), (1, 0), (1, 2), (2, 1)])
searches = [
    ("minimax (reference)", reference.minimax_search, TIC_TAC_TOE),
    ("alpha-beta", algorithms.alpha_beta_search, TIC_TAC_TOE),
    ("alpha-beta, centre and corners first", algorithms.alpha_beta_search, centre_first),
    ("alpha-beta, strict cutoffs (reference)", strict_alpha_beta, TIC_TAC_TOE),
]
visited = {}
print(f"{'':<40}{'value':>6}{'positions':>11}{'seconds':>9}")
for label, search, game in searches:
    (value, _), visited[label], seconds = run(search, game, game.initial)
    print(f"{label:<40}{value:>6}{visited[label]:>11}{seconds:>9.2f}")

**Do przemyślenia.** Czy wszystkie cztery przeszukiwania podały tę samą wartość? Dlaczego kolejność „najpierw środek i rogi” pomaga? Dlaczego nierówności ostre kosztują w kółku i krzyżyku tak dużo?

## Eksperyment 2: 200 losowych kolejności pól

Dla każdego z 200 ziaren losujemy kolejność dziewięciu pól i liczymy, ile pozycji odwiedza alfa-beta z pustej planszy. Histogram pokazuje, jak często wychodzi dana liczba pozycji. Linie przerywane to kolejność naturalna (wiersz po wierszu) i „najpierw środek i rogi”. Obliczenia trwają około pół minuty.

**Twoje przewidywanie.** Ile pozycji odwiedzi alfa-beta przy najlepszej i przy najgorszej z losowych kolejności? Gdzie na histogramie wypadnie kolejność naturalna?

*Moje przewidywanie:* …

In [ ]:
counts = []
for seed in range(200):
    cells = CELLS[:]
    random.Random(seed).shuffle(cells)
    game = ordered(cells)
    _, positions, _ = run(algorithms.alpha_beta_search, game, game.initial)
    counts.append(positions)
print(f"fewest {min(counts)}, most {max(counts)}, mean {mean(counts):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(counts, bins=30, color="#2a78d6", edgecolor="#fcfcfb", linewidth=1)
markers = [
    ("natural order", visited["alpha-beta"], "#eb6834"),
    ("centre and corners first", visited["alpha-beta, centre and corners first"], "#1baf7a"),
]
for label, x, color in markers:
    ax.axvline(x, color=color, linewidth=2, linestyle="--", label=f"{label}: {x}")
ax.set_title(
    "Positions alpha-beta visits from the empty board, 200 random cell orders",
    loc="left",
    color="#0b0b0b",
)
ax.set_xlabel("positions visited", color="#52514e")
ax.set_ylabel("cell orders", color="#52514e")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(frameon=False, loc="upper right", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Ile razy więcej pozycji odwiedza najgorsza z losowych kolejności niż najlepsza? Czy któraś losowa kolejność pobiła „najpierw środek i rogi”? Co to mówi o tym, jak ważna jest kolejność ruchów w grach, w których drzewo jest o wiele większe?

## Eksperyment 3: oszczędność a liczba wolnych pól

Dla każdej liczby wolnych pól od 2 do 8 losujemy 10 pozycji (z ziarnem, przez losowe ruchy) i liczymy średnio, ile pozycji odwiedzają z nich minimax i Twój alfa-beta. Przy 9 wolnych polach jest tylko pusta plansza z eksperymentu 1. Linia przerywana to pierwiastek z liczby pozycji minimaksu, czyli to, czego można się spodziewać przy idealnej kolejności ruchów. Oś pionowa ma skalę logarytmiczną.

**Twoje przewidywanie.** Przy ilu wolnych polach alfa-beta przestaje cokolwiek oszczędzać? Jak zmienia się oszczędność, gdy wolnych pól przybywa?

*Moje przewidywanie:* …

In [ ]:
sizes = list(range(2, 10))
minimax_mean, alpha_beta_mean = [], []
for empty in sizes:
    if empty == 9:
        minimax_mean.append(visited["minimax (reference)"])
        alpha_beta_mean.append(visited["alpha-beta"])
        continue
    positions, seed = [], 0
    while len(positions) < 10:
        state = random_position(empty, 1000 * empty + seed)
        seed += 1
        if state is not None and empty_cells(state) == empty:
            positions.append(state)
    minimax_mean.append(mean(run(reference.minimax_search, TIC_TAC_TOE, s)[1] for s in positions))
    alpha_beta_mean.append(
        mean(run(algorithms.alpha_beta_search, TIC_TAC_TOE, s)[1] for s in positions)
    )
print(f"{'empty cells':<13}{'minimax':>10}{'alpha-beta':>12}{'ratio':>8}")
for empty, m, a in zip(sizes, minimax_mean, alpha_beta_mean):
    print(f"{empty:<13}{m:>10.0f}{a:>12.0f}{a / m:>8.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for label, values, color in (
    ("minimax", minimax_mean, "#2a78d6"),
    ("alpha-beta", alpha_beta_mean, "#eb6834"),
):
    ax.plot(
        sizes,
        values,
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.plot(
    sizes,
    [m**0.5 for m in minimax_mean],
    color="#52514e",
    linewidth=2,
    linestyle="--",
    label="square root of minimax (perfect order)",
)
ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_title(
    "Positions visited, by the number of empty cells (log scale)", loc="left", color="#0b0b0b"
)
ax.set_xlabel("empty cells", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Dlaczego przy dwóch wolnych polach alfa-beta nie ma czego przycinać? Jak szybko rośnie przewaga alfa-beta z każdym kolejnym wolnym polem? Czy Twój alfa-beta zbliża się do linii przerywanej, czy raczej do minimaksu?